# Fintech Data Pipeline
**Student:** Paras Dwivedi | **ID:** bitsom_ftai_2601265

This notebook covers:
- Part 3: Python Reconciliation Workflow
- Part 4: JSON Normalization
- Dashboard Data Exports

## Part 3: Python Reconciliation Workflow

### Step 1 — Load Both Files

In [ ]:
import pandas as pd
import json

# Load ledger and gateway files
ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')

print('Ledger shape:', ledger.shape)
print('Gateway shape:', gateway.shape)
print('\nLedger preview:')
print(ledger.head())
print('\nGateway preview:')
print(gateway.head())

### Step 2 — Check Duplicates and Nulls

In [ ]:
print('=== LEDGER ===')
print('Duplicates:', ledger.duplicated().sum())
print('Nulls:')
print(ledger.isnull().sum())

print('\n=== GATEWAY ===')
print('Duplicates:', gateway.duplicated().sum())
print('Nulls:')
print(gateway.isnull().sum())

### Step 3 — Records Missing in Gateway

In [ ]:
missing_in_gateway = ledger[~ledger['transaction_id'].isin(gateway['transaction_id'])]
print('Missing in gateway:', len(missing_in_gateway))
print(missing_in_gateway)
missing_in_gateway.to_csv('missing_in_gateway.csv', index=False)

### Step 4 — Records Missing in Ledger

In [ ]:
missing_in_ledger = gateway[~gateway['transaction_id'].isin(ledger['transaction_id'])]
print('Missing in ledger:', len(missing_in_ledger))
print(missing_in_ledger)
missing_in_ledger.to_csv('missing_in_ledger.csv', index=False)

### Step 5 — Amount Mismatches

In [ ]:
merged = ledger.merge(gateway, on='transaction_id', suffixes=('_ledger', '_gateway'))
amount_mismatches = merged[merged['amount_usd_ledger'] != merged['amount_usd_gateway']]
print('Amount mismatches:', len(amount_mismatches))
print(amount_mismatches[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway']])
amount_mismatches.to_csv('amount_mismatches.csv', index=False)

### Step 6 — Status Mismatches

In [ ]:
status_mismatches = merged[merged['status_ledger'] != merged['status_gateway']]
print('Status mismatches:', len(status_mismatches))
print(status_mismatches[['transaction_id', 'status_ledger', 'status_gateway']])
status_mismatches.to_csv('status_mismatches.csv', index=False)

### Step 7 — Final Reconciliation Report

In [ ]:
all_ids = pd.concat([ledger[['transaction_id']], gateway[['transaction_id']]]).drop_duplicates()

report_rows = []
for tid in all_ids['transaction_id']:
    l = ledger[ledger['transaction_id'] == tid]
    g = gateway[gateway['transaction_id'] == tid]
    
    l_amt = l['amount_usd'].values[0] if len(l) > 0 else None
    g_amt = g['amount_usd'].values[0] if len(g) > 0 else None
    l_status = l['status'].values[0] if len(l) > 0 else None
    g_status = g['status'].values[0] if len(g) > 0 else None
    
    if len(l) == 0:
        recon_status = 'missing_in_ledger'
    elif len(g) == 0:
        recon_status = 'missing_in_gateway'
    elif l_amt != g_amt:
        recon_status = 'amount_mismatch'
    elif l_status != g_status:
        recon_status = 'status_mismatch'
    else:
        recon_status = 'matched'
    
    report_rows.append({
        'transaction_id': tid,
        'ledger_amount': l_amt,
        'gateway_amount': g_amt,
        'ledger_status': l_status,
        'gateway_status': g_status,
        'reconciliation_status': recon_status
    })

recon_report = pd.DataFrame(report_rows)
print(recon_report)
recon_report.to_csv('reconciliation_report.csv', index=False)

### Step 8 — Summary Metrics

In [ ]:
issue_ids = recon_report[recon_report['reconciliation_status'] != 'matched']['transaction_id']
amount_at_risk = recon_report[recon_report['transaction_id'].isin(issue_ids)]['ledger_amount'].sum()

summary = {
    'total_ledger_rows': len(ledger),
    'total_gateway_rows': len(gateway),
    'missing_in_gateway_count': len(missing_in_gateway),
    'missing_in_ledger_count': len(missing_in_ledger),
    'amount_mismatch_count': len(amount_mismatches),
    'status_mismatch_count': len(status_mismatches),
    'reconciliation_issue_count': len(recon_report[recon_report['reconciliation_status'] != 'matched']),
    'ledger_total_amount': round(ledger['amount_usd'].sum(), 2),
    'gateway_total_amount': round(gateway['amount_usd'].sum(), 2),
    'amount_at_risk': round(float(amount_at_risk), 2)
}

print(json.dumps(summary, indent=2))
with open('summary_metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\nsummary_metrics.json saved!')

---
## Part 4: JSON Normalization

### Step 1 — Read JSON File

In [ ]:
with open('api_response_sample.json', 'r') as f:
    api_data = json.load(f)

print('Top-level keys:', list(api_data.keys()))
print('Number of batches:', len(api_data['batches']))

### Step 2 — Flatten Nested JSON

In [ ]:
rows = []
for batch in api_data['batches']:
    for settlement in batch['settlements']:
        rows.append({
            'batch_id': batch['batch_id'],
            'merchant_id': batch['merchant']['merchant_id'],
            'merchant_name': batch['merchant']['merchant_name'],
            'region': batch['merchant']['region'],
            'settlement_id': settlement['settlement_id'],
            'amount_usd': settlement['amount_usd'],
            'status': settlement['status'],
            'processed_at': settlement['processed_at'],
            'bank_name': settlement['bank']['name'],
            'bank_country': settlement['bank']['country']
        })

api_df = pd.DataFrame(rows)
print('Shape:', api_df.shape)
print(api_df)

### Step 3 — Clean Column Names & Convert Datetime

In [ ]:
api_df.columns = [col.lower().replace(' ', '_') for col in api_df.columns]
api_df['processed_at'] = pd.to_datetime(api_df['processed_at'])
print(api_df.dtypes)
print(api_df)

### Step 4 — Save api_normalized.csv

In [ ]:
api_df.to_csv('api_normalized.csv', index=False)
print('api_normalized.csv saved! Shape:', api_df.shape)

---
## Dashboard Data Exports

### Load Cleaned Transactions

In [ ]:
ct = pd.read_csv('cleaned_transactions.csv')
ct['transaction_date'] = pd.to_datetime(ct['transaction_date'])
captured = ct[ct['status'] == 'captured']
print('Total rows:', len(ct))
print('Captured rows:', len(captured))

### Daily Summary

In [ ]:
daily = ct.groupby('transaction_date').agg(
    total_gmv_usd=('amount_usd', 'sum'),
    transaction_count=('transaction_id', 'count'),
    successful_count=('status', lambda x: (x == 'captured').sum())
).reset_index()
print(daily)
daily.to_csv('daily_summary.csv', index=False)

### Payment Method Breakdown

In [ ]:
pm = captured.groupby('payment_method').agg(
    total_gmv_usd=('amount_usd', 'sum'),
    transaction_count=('transaction_id', 'count')
).reset_index()
print(pm)
pm.to_csv('payment_method_breakdown.csv', index=False)

### Region Breakdown

In [ ]:
rb = ct.groupby('default_region').agg(
    total_gmv_usd=('amount_usd', 'sum'),
    transaction_count=('transaction_id', 'count'),
    avg_risk_score=('risk_score', 'mean')
).reset_index()
print(rb)
rb.to_csv('region_breakdown.csv', index=False)

### Merchant Performance Summary

In [ ]:
mp = ct.groupby('merchant_name').agg(
    total_gmv_usd=('amount_usd', 'sum'),
    captured_gmv_usd=('amount_usd', lambda x: x[ct.loc[x.index, 'status'] == 'captured'].sum()),
    chargeback_count=('status', lambda x: (x == 'chargeback').sum()),
    high_risk_count=('high_risk_flag', 'sum')
).reset_index()
print(mp)
mp.to_csv('merchant_performance_summary.csv', index=False)
print('\nAll dashboard CSVs saved!')